In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

### Data Exploration - Double Check - Done

In [21]:
# Load the dataset
orders = pd.read_csv('data/orders.csv', parse_dates=['order_date'])
customers = pd.read_csv('data/customers.csv')
products = pd.read_csv('data/products.csv')

In [22]:
# Q1. Trung vi khoang cach giua 2 don hang lien tiep cua moi khach hang la bao nhieu?
orders['order_date'] = pd.to_datetime(orders['order_date'])
# Sap xep theo khach hang va ngay mua
orders = orders.sort_values(by=['customer_id', 'order_date'])
# TInh khoang cach giua cac lan mua
orders['gap'] = orders.groupby('customer_id')['order_date'].diff().dt.days
# Tinh trung vi
median_gap = orders['gap'].median()
print(f'Trung vi khoang cach giua 2 don hang lien tiep cua moi khach hang la: {median_gap} ngay')

Trung vi khoang cach giua 2 don hang lien tiep cua moi khach hang la: 144.0 ngay


In [23]:
# Q2. Phan khuc san pham (segment) nao trong products.csv co ty suat loi nhuan gop trung binh cao nhat?
# cong thuc: (price - cogs) / price
products['margin'] = (products['price'] - products['cogs']) / products['price']
highest_margin_segment = products.groupby('segment')['margin'].mean().idxmax()
print(f'Phan khuc san pham co ty suat loi nhuan gop trung binh cao nhat la: {highest_margin_segment}')

Phan khuc san pham co ty suat loi nhuan gop trung binh cao nhat la: Standard


In [24]:
# Q3: Ly do tra hang pho bien nhat cho san pham Streetwear la gi?
returns = pd.read_csv('data/returns.csv')
returns_products = returns.merge(products, on='product_id')
streetwear_returns = returns_products[returns_products['category'] == 'Streetwear']
top_reason = streetwear_returns['return_reason'].value_counts().idxmax()
print(f'Ly do tra hang pho bien nhat cho san pham Streetwear la: {top_reason}')

Ly do tra hang pho bien nhat cho san pham Streetwear la: wrong_size


In [25]:
# Q4: Nguon truy cap co ty le thoat (bounce rate) thap nhat
traffic = pd.read_csv('data/web_traffic.csv')
lowest_bounce = traffic.groupby('traffic_source')['bounce_rate'].mean().idxmin()
print(f'Nguon truy cap co ty le thoat thap nhat la: {lowest_bounce}')

Nguon truy cap co ty le thoat thap nhat la: email_campaign


In [26]:
# Q5: Ty le dong ap dung khuyen mai trong order_items.csv la bao nhieu?
order_items = pd.read_csv('data/order_items.csv')
promo_applied = order_items['promo_id'].notna().sum()
total_rows = len(order_items)
percentage = (promo_applied / total_rows) * 100
print(f'Ty le dong ap dung khuyen mai trong order_items.csv la: {percentage:.2f}%')

Ty le dong ap dung khuyen mai trong order_items.csv la: 38.66%


/tmp/ipykernel_226564/885313789.py:2: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv('data/order_items.csv')


In [27]:
# Q6: Nhom tuoi co so don trung binh tren moi khach cao nhat la nhom nao? (tong so don/so kahch trong nhom)
customers = pd.read_csv('data/customers.csv')
customers = customers.dropna(subset=['age_group'])
# Đếm số đơn của mỗi user
user_order_counts = orders.groupby('customer_id').size().reset_index(name='order_count')
# Join lại với customers
cust_orders = customers.merge(user_order_counts, on='customer_id', how='left').fillna(0)
# Tính trung bình đơn hàng theo nhóm tuổi
avg_orders = cust_orders.groupby('age_group')['order_count'].mean()
top_age_group = avg_orders.idxmax()
print(f'Nhom tuoi co so don trung binh tren moi khach cao nhat la: {top_age_group}')

Nhom tuoi co so don trung binh tren moi khach cao nhat la: 55+


In [28]:
# Q7:  Vung (region)nao co doanh thu cao nhat? (tu tinh revenue bang cach join orders.csv (lay cot zip), order_items.csv (de tinh tien: quantity * unit_price - discount_amount), va geography.csv (lay cot region))
geo = pd.read_csv('data/geography.csv')
order_items['item_revenue'] = (order_items['quantity'] * order_items['unit_price']) - order_items['discount_amount'].fillna(0)
order_revenue = order_items.groupby('order_id')['item_revenue'].sum().reset_index()
# join orders de lay zip, roi join geo de lay region
orders_rev = orders.merge(order_revenue, on='order_id')
orders_geo = orders_rev.merge(geo, on='zip')
top_region = orders_geo.groupby('region')['item_revenue'].sum().idxmax()
print(f'Vung co doanh thu cao nhat la: {top_region}')

Vung co doanh thu cao nhat la: East


In [29]:
# Q8: Phuong thuc thanh toan nao nhieu nhat cho don huy?
cancelled = orders[orders['order_status'] == 'cancelled']
top_payment_method = cancelled['payment_method'].value_counts().idxmax()
print(f'Phuong thuc thanh toan nhieu nhat cho don huy la: {top_payment_method}')

Phuong thuc thanh toan nhieu nhat cho don huy la: credit_card


In [30]:
# Q9: Kich thuoc (size) co ty le tra hang cao nhat? (so ban ghi trong returns chia cho so dong trong order_items (join voi products theo product_id))
# Dem so luong ban ra theo size
sales_by_size = order_items.merge(products, on='product_id')['size'].value_counts()
# Dem so luong tra ve theo size
returns_by_size = returns.merge(products, on='product_id')['size'].value_counts()
# Tinh ty le tra hang
return_rate = (returns_by_size / sales_by_size).fillna(0)
top_size = return_rate.idxmax()
print(f'Kich thuoc co ty le tra hang cao nhat la: {top_size}')


Kich thuoc co ty le tra hang cao nhat la: S


In [31]:
# Q10: Ky tra gop co gia tri thanh toan trung binh cao nhat?
payments = pd.read_csv('data/payments.csv')
top_installment = payments.groupby('installments')['payment_value'].mean().idxmax()
print(f'Ky tra gop co gia tri thanh toan trung binh cao nhat la: {top_installment} ky')

Ky tra gop co gia tri thanh toan trung binh cao nhat la: 6 ky
